# 📃 Generate Data for tables

## 🚗 Generate Drivers

In [2]:
import random
import pandas as pd
from faker import Faker
from datetime import date

fake = Faker()

N_DRIVERS = 50_000
cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"]
vehicle_types = ["economy", "premium", "XL"]
statuses = ["active", "inactive", "suspended"]

drivers = []
for i in range(1, N_DRIVERS + 1):
    drivers.append({
        "driver_id": i,
        "name": fake.name(),
        "signup_date": fake.date_between(start_date="-4y", end_date="-6m"),
        "city": random.choice(cities),
        "vehicle_type": random.choices(vehicle_types, weights=[0.6, 0.25, 0.15])[0],
        "status": random.choices(statuses, weights=[0.8, 0.15, 0.05])[0]
    })

drivers_df = pd.DataFrame(drivers)


## 🧍 Generate Riders

In [3]:
N_RIDERS = 200_000
segments = ["casual", "power", "corporate"]

riders = []
for i in range(1, N_RIDERS + 1):
    riders.append({
        "rider_id": i,
        "name": fake.name(),
        "signup_date": fake.date_between(start_date="-3y", end_date="-1m"),
        "home_city": random.choice(cities),
        "segment": random.choices(segments, weights=[0.55, 0.3, 0.15])[0]
    })

riders_df = pd.DataFrame(riders)


## 🚕 Generate Trips (BIG TABLE)

In [4]:
from tqdm import trange
import numpy as np
from datetime import timedelta

N_TRIPS = 3_000_000

trips = []
for i in trange(1, N_TRIPS + 1):
    rider_id = random.randint(1, N_RIDERS)
    driver_id = random.randint(1, N_DRIVERS)

    request_time = fake.date_time_between(start_date="-1y", end_date="now")
    pickup_delay = random.choice([None, random.randint(1, 10)])
    pickup_time = request_time + timedelta(minutes=pickup_delay) if pickup_delay else None

    duration = random.uniform(5, 45)
    dropoff_time = pickup_time + timedelta(minutes=duration) if pickup_time else None

    distance = duration * random.uniform(0.4, 0.8)
    surge = random.choices([1.0, 1.2, 1.5, 2.0], weights=[0.7, 0.15, 0.1, 0.05])[0]

    base_fare = distance * random.uniform(10, 18)
    fare = round(base_fare * surge, 2)
    platform_fee = round(fare * random.uniform(0.18, 0.25), 2)
    driver_earnings = round(fare - platform_fee, 2)

    status = random.choices(
        ["completed", "rider_cancelled", "driver_cancelled"],
        weights=[0.85, 0.1, 0.05]
    )[0]

    trips.append({
        "trip_id": i,
        "rider_id": rider_id,
        "driver_id": driver_id,
        "request_time": request_time,
        "pickup_time": pickup_time,
        "dropoff_time": dropoff_time,
        "pickup_city": random.choice(cities),
        "dropoff_city": random.choice(cities),
        "status": status,
        "distance_km": round(distance, 2),
        "duration_min": round(duration, 2),
        "surge_multiplier": surge,
        "fare_amount": fare if status == "completed" else 0,
        "driver_earnings": driver_earnings if status == "completed" else 0,
        "platform_fee": platform_fee if status == "completed" else 0
    })

trips_df = pd.DataFrame(trips)


100%|██████████| 3000000/3000000 [01:45<00:00, 28448.16it/s]


## ⭐ Generate Ratings

In [6]:
import numpy as np
import pandas as pd
import random

trips_df = pd.read_csv("dataset/trips.csv")
# 1️⃣ Filter completed trips and drop NULL-critical columns
completed_trips = (
    trips_df
    .loc[trips_df["status"] == "completed"]
    .dropna(subset=["trip_id", "dropoff_time"])
)

# 2️⃣ Ensure we have enough rows
sample_size = min(2_000_000, len(completed_trips))

completed_trips = completed_trips.sample(
    n=sample_size,
    random_state=42
)

# 3️⃣ Generate ratings safely (vectorized)
ratings_df = pd.DataFrame({
    "rating_id": np.arange(1, sample_size + 1),
    "trip_id": completed_trips["trip_id"].values,
    "rated_by": np.random.choice(
        ["rider", "driver"],
        size=sample_size
    ),
    "rating": np.random.choice(
        [3, 4, 5],
        size=sample_size,
        p=[0.1, 0.4, 0.5]
    ),
    "rating_timestamp": completed_trips["dropoff_time"].values
})

# 4️⃣ Final NULL safety check (assertion)
assert ratings_df.isnull().sum().sum() == 0



In [8]:
ratings_df.to_csv("dataset/ratings.csv", index = False)

## 💰 Driver Payouts

In [9]:
import numpy as np
import pandas as pd

# 1️⃣ Ensure datetime
trips_df["request_time"] = pd.to_datetime(trips_df["request_time"])

# 2️⃣ Add payout period (monthly)
trips_df["payout_period_start"] = trips_df["request_time"].dt.to_period("M").dt.start_time
trips_df["payout_period_end"] = trips_df["request_time"].dt.to_period("M").dt.end_time

# 3️⃣ Aggregate COMPLETED trips
completed_payouts = (
    trips_df[trips_df["status"] == "completed"]
    .dropna(subset=["driver_id", "driver_earnings"])
    .groupby(["driver_id", "payout_period_start", "payout_period_end"])
    .agg(
        total_trips=("trip_id", "count"),
        total_earnings=("driver_earnings", "sum")
    )
    .reset_index()
)

# 4️⃣ Aggregate cancellations
cancellations = (
    trips_df[trips_df["status"] != "completed"]
    .groupby(["driver_id", "payout_period_start", "payout_period_end"])
    .size()
    .reset_index(name="cancellations")
)

# 5️⃣ Merge cancellations
driver_payouts_df = completed_payouts.merge(
    cancellations,
    on=["driver_id", "payout_period_start", "payout_period_end"],
    how="left"
)

driver_payouts_df["cancellations"] = driver_payouts_df["cancellations"].fillna(0).astype(int)

# 6️⃣ Bonuses (numeric(10,2))
driver_payouts_df["bonuses"] = np.where(
    driver_payouts_df["total_trips"] > 300,
    500.00,
    0.00
)

# 7️⃣ Enforce NOT NULL + rounding
driver_payouts_df["total_trips"] = driver_payouts_df["total_trips"].astype(int)
driver_payouts_df["total_earnings"] = driver_payouts_df["total_earnings"].round(2)
driver_payouts_df["bonuses"] = driver_payouts_df["bonuses"].round(2)

# 8️⃣ Generate primary key
driver_payouts_df = driver_payouts_df.reset_index(drop=True)
driver_payouts_df["payout_id"] = driver_payouts_df.index + 1

# 9️⃣ Final column order (matches SQL schema)
driver_payouts_df = driver_payouts_df[
    [
        "payout_id",
        "driver_id",
        "payout_period_start",
        "payout_period_end",
        "total_trips",
        "total_earnings",
        "bonuses",
        "cancellations"
    ]
]

# 🔒 Final safety check
assert driver_payouts_df.isnull().sum().sum() == 0



In [10]:
# drivers_df.to_csv("dataset/drivers.csv", index=False)
# riders_df.to_csv("dataset/riders.csv", index=False)
# trips_df.to_csv("dataset/trips.csv", index=False)
# ratings_df.to_csv("dataset/ratings.csv", index=False)
driver_payouts_df.to_csv("dataset/driver_payouts.csv", index=False)

In [11]:
import numpy as np
import pandas as pd

driver_payouts_df = pd.read_csv("dataset/driver_payouts.csv")
driver_payouts_df

,payout_id,driver_id,payout_period_start,payout_period_end,total_trips,total_earnings,bonuses,cancellations
0,1,1,2025-02-01,2025-02-28 23:59:59.999999,3,816.17,0.0,2
1,2,1,2025-03-01,2025-03-31 23:59:59.999999,5,1007.63,0.0,0
2,3,1,2025-04-01,2025-04-30 23:59:59.999999,3,550.06,0.0,1
3,4,1,2025-05-01,2025-05-31 23:59:59.999999,2,396.84,0.0,1
4,5,1,2025-06-01,2025-06-30 23:59:59.999999,6,1665.84,0.0,1
...,...,...,...,...,...,...,...,...
613345,613346,50000,2025-09-01,2025-09-30 23:59:59.999999,7,1259.60,0.0,1
613346,613347,50000,2025-10-01,2025-10-31 23:59:59.999999,6,886.08,0.0,0
613347,613348,50000,2025-11-01,2025-11-30 23:59:59.999999,7,1041.98,0.0,0
613348,613349,50000,2025-12-01,2025-12-31 23:59:59.999999,3,342.69,0.0,0
